In [ ]:
# %%writefile 04_test_model.ipynb
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Test & Evaluate Model\n",
    "# Comprehensive testing of code and chat capabilities"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install -q transformers accelerate"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "from transformers import AutoModelForCausalLM, AutoTokenizer\n",
    "import time\n",
    "\n",
    "# CONFIG\n",
    "MODEL_PATH = \"./production-model-merged\"  # or \"./codellama-coding-chat-final\"\n",
    "USE_PEFT = False  # Set True if using LoRA adapter\n",
    "\n",
    "# Load\n",
    "print(\"Loading model...\")\n",
    "tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)\n",
    "model = AutoModelForCausalLM.from_pretrained(\n",
    "    MODEL_PATH,\n",
    "    torch_dtype=torch.float16,\n",
    "    device_map=\"auto\"\n",
    ")\n",
    "model.eval()\n",
    "print(\"✅ Loaded\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# TEST SUITE\n",
    "code_tests = [\n",
    "    {\n",
    "        \"name\": \"Python Function\",\n",
    "        \"prompt\": \"Write a Python function to check if a number is prime\",\n",
    "        \"keywords\": [\"def\", \"return\", \"prime\"]\n",
    "    },\n",
    "    {\n",
    "        \"name\": \"SQL Query\",\n",
    "        \"prompt\": \"Write a SQL query to find the top 5 customers by total purchase amount\",\n",
    "        \"keywords\": [\"SELECT\", \"FROM\", \"ORDER BY\", \"LIMIT\"]\n",
    "    },\n",
    "    {\n",
    "        \"name\": \"Algorithm\",\n",
    "        \"prompt\": \"Explain binary search algorithm with Python code\",\n",
    "        \"keywords\": [\"binary\", \"search\", \"mid\", \"while\"]\n",
    "    },\n",
    "    {\n",
    "        \"name\": \"Debug Help\",\n",
    "        \"prompt\": \"My Python code has an IndexError. How do I fix it?\",\n",
    "        \"keywords\": [\"index\", \"range\", \"len\", \"check\"]\n",
    "    }\n",
    "]\n",
    "\n",
    "chat_tests = [\n",
    "    {\n",
    "        \"name\": \"Explanation\",\n",
    "        \"prompt\": \"Explain machine learning to a 10 year old\",\n",
    "        \"check\": \"simple language\"\n",
    "    },\n",
    "    {\n",
    "        \"name\": \"Advice\",\n",
    \"prompt\": \"How do I stay motivated while learning to code?\",\n",
    "        \"check\": \"encouraging\"\n",
    "    }\n",
    "]\n",
    "\n",
    "def run_test(prompt, max_tokens=300):\n",
    "    formatted = f\"[INST] {prompt} [/INST]\"\n",
    "    inputs = tokenizer(formatted, return_tensors=\"pt\").to(\"cuda\")\n",
    "    \n",
    "    start = time.time()\n",
    "    with torch.no_grad():\n",
    "        outputs = model.generate(\n",
    "            **inputs,\n",
    "            max_new_tokens=max_tokens,\n",
    "            temperature=0.7,\n",
    "            do_sample=True,\n",
    "            top_p=0.9,\n",
    "        )\n",
    "    elapsed = time.time() - start\n",
    "    \n",
    "    response = tokenizer.decode(outputs[0], skip_special_tokens=True)\n",
    "    response = response.replace(formatted, \"\").strip()\n",
    "    \n",
    "    tokens = len(outputs[0]) - len(inputs[0])\n",
    "    speed = tokens / elapsed\n",
    "    \n",
    "    return response, elapsed, speed"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# RUN CODE TESTS\n",
    "print(\"=\" * 60)\n",
    "print(\"CODE TESTS\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "for test in code_tests:\n",
    "    print(f\"\\n📝 {test['name']}\")\n",
    "    print(f\"Prompt: {test['prompt']}\")\n    \n",
    "    response, elapsed, speed = run_test(test['prompt'])\n",
    "    \n",
    "    # Check keywords\n",
    "    found = [k for k in test['keywords'] if k.lower() in response.lower()]\n",
    "    score = len(found) / len(test['keywords'])\n",
    "    \n",
    "    print(f\"Response: {response[:200]}...\")\n",
    "    print(f\"⏱️  Time: {elapsed:.2f}s | Speed: {speed:.1f} tokens/sec\")\n",
    "    print(f\"✅ Score: {score*100:.0f}% ({len(found)}/{len(test['keywords'])} keywords)\")\n",
    "    print(\"-\" * 60)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# RUN CHAT TESTS\n",
    "print(\"\\n\" + \"=\" * 60)\n",
    "print(\"CHAT TESTS\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "for test in chat_tests:\n",
    "    print(f\"\\n💬 {test['name']}\")\n",
    "    print(f\"Prompt: {test['prompt']}\")\n",
    "    \n",
    "    response, elapsed, speed = run_test(test['prompt'], max_tokens=400)\n",
    "    \n",
    "    print(f\"Response: {response[:250]}...\")\n",
    "    print(f\"⏱️  Time: {elapsed:.2f}s | Speed: {speed:.1f} tokens/sec\")\n",
    "    print(\"-\" * 60)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# SPEED BENCHMARK\n",
    "print(\"\\n\" + \"=\" * 60)\n",
    "print(\"SPEED BENCHMARK\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "prompt_lengths = [50, 100, 200]\n",
    "output_lengths = [100, 200, 500]\n",
    "\n",
    "base_prompt = \"Explain how Python dictionaries work\"\n",
    "\n",
    "for out_len in output_lengths:\n",
    "    _, elapsed, speed = run_test(base_prompt, max_tokens=out_len)\n",
    "    print(f\"Output {out_len} tokens: {elapsed:.2f}s ({speed:.1f} tok/sec)\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}